In [13]:
import requests
from bs4 import BeautifulSoup
import re
from collections import Counter, defaultdict
from tqdm import tqdm

def extract_text(url):
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")

        for tag in soup(["script", "style", "noscript"]):
            tag.decompose()

        text = soup.get_text(separator=" ")
        return text
    except:
        return ""

def clean_text(text):
    text = text.lower()
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[^a-z0-9\s]', '', text)
    return text.strip()

In [14]:
def build_corpus(urls):
    texts = []
    for url in tqdm(urls):
        raw = extract_text(url)
        if raw:
            cleaned = clean_text(raw)
            if len(cleaned) > 200:
                texts.append(cleaned)
    return list(set(texts))  # deduplicate

In [6]:
def prepare_words(texts):
    words = []
    for text in texts:
        words.extend(text.split())
    return words

In [7]:
def get_vocab(words):
    vocab = Counter()
    for word in words:
        chars = ' '.join(list(word)) + ' </w>'
        vocab[chars] += 1
    return vocab

In [8]:
def get_pair_stats(vocab):
    pairs = defaultdict(int)
    for word, freq in vocab.items():
        symbols = word.split()
        for i in range(len(symbols) - 1):
            pairs[(symbols[i], symbols[i+1])] += freq
    return pairs

In [9]:
def merge_vocab(pair, vocab):
    new_vocab = {}
    bigram = ' '.join(pair)
    replacement = ''.join(pair)

    for word in vocab:
        new_word = word.replace(bigram, replacement)
        new_vocab[new_word] = vocab[word]

    return new_vocab

In [10]:
def train_bpe(words, num_merges=1000):
    vocab = get_vocab(words)
    merges = []

    for i in range(num_merges):
        pairs = get_pair_stats(vocab)
        if not pairs:
            break

        best_pair = max(pairs, key=pairs.get)
        vocab = merge_vocab(best_pair, vocab)
        merges.append(best_pair)

        print(f"Merge {i+1}: {best_pair}")

    return merges, vocab

In [11]:
def tokenize_word(word, merges):
    tokens = list(word) + ['</w>']

    for pair in merges:
        i = 0
        while i < len(tokens) - 1:
            if (tokens[i], tokens[i+1]) == pair:
                tokens[i:i+2] = [''.join(pair)]
            else:
                i += 1
    return tokens

In [15]:
if __name__ == "__main__":

    urls = [
        "https://en.wikipedia.org/wiki/Machine_learning",
        "https://en.wikipedia.org/wiki/Artificial_intelligence",
        "https://en.wikipedia.org/wiki/Neural_network"
    ]

    texts = build_corpus(urls)
    words = prepare_words(texts)

    merges, vocab = train_bpe(words, num_merges=500)

    print("\nTokenizing example:")
    print(tokenize_word("transformer", merges))

100%|████████████████████████████████████████████████████████████████████████████████████| 3/3 [00:01<00:00,  2.39it/s]


Tokenizing example:
['t', 'r', 'a', 'n', 's', 'f', 'o', 'r', 'm', 'e', 'r', '</w>']


In [18]:
print(tokens)

NameError: name 'tokens' is not defined